### Estudo de Caso - Data's Light Academy

![case_1](https://github.com/ajluz/public_images/blob/main/workshop_dlt_sql/case_1.png?raw=true)

### Requisitos do Projeto da Data's Light Academy

![case_1](https://github.com/ajluz/public_images/blob/main/workshop_dlt_sql/case_2.png?raw=true)

### Data's light Academy + Databricks
![case_1](https://github.com/ajluz/public_images/blob/main/workshop_dlt_sql/case_3.png?raw=true)


### Databricks como Plataforma de Data Intelligence
![case_1](https://learn.microsoft.com/en-us/azure/databricks/_static/images/lakehouse-architecture/ref-arch-overview-azure.png)

### Entendendo a Medallion Architecture

![case_1](https://github.com/ajluz/public_images/blob/main/workshop_dlt_sql/medallion_1.png?raw=true)

### Case de implementação da Data's Light Academy
![case_1](https://github.com/ajluz/public_images/blob/main/workshop_dlt_sql/medallion_3.png?raw=true)

### Setup

In [0]:
%run "./functions"

In [0]:
generate_files_data(100000)
generate_database_data(type_of_data='initial setup')
query = generate_api_data()

### Dados na Zona Bronze

In [0]:
%sql
SELECT * FROM dataslight_dev.bronze.dados_database

In [0]:
%sql
SELECT * FROM dataslight_dev.bronze.dados_files

In [0]:
%sql
SELECT * FROM dataslight_dev.bronze.dados_api
WHERE 1=1
-- -- QUARENTENA - Tabela para analises futuras
-- AND payload.info_produto.product_name = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
--   OR payload.info_pagamento.valor = '0.00'
--   OR payload.info_pagamento.forma_pagamento = '-----'

-- -- Constraint de Check e Drop para Clientes
-- AND payload.info_usuario.email like '%#%'
  -- OR payload.info_usuario.sexo like '%X%'

-- -- Constraint de Check e Drop para Clientes
-- AND ip_address = '999.999.999.999'
  -- OR access_point = 'unknown'

### Dados na Zona Silver

In [0]:
%sql SELECT * FROM dataslight_dev.silver.tb_quarentena

In [0]:
%sql SELECT * FROM dataslight_dev.silver.tb_acesso

In [0]:
%sql SELECT * FROM dataslight_dev.silver.tb_usuario

In [0]:
%sql SELECT * FROM dataslight_dev.silver.tb_venda

In [0]:
%sql SELECT * FROM dataslight_dev.silver.tb_curso

In [0]:
spark.sql('SELECT * FROM dataslight_dev.bronze.dados_database').display()
spark.sql('SELECT * FROM dataslight_dev.silver.tb_curso').display()

In [0]:
generate_database_data(type_of_data='update row')

In [0]:
spark.sql('SELECT * FROM dataslight_dev.bronze.dados_database').display()
spark.sql('SELECT * FROM dataslight_dev.silver.tb_curso').display()

### Dados na Zona Gold

In [0]:
%sql SELECT * FROM dataslight_dev.gold.dim_codigos

In [0]:
%sql SELECT * FROM dataslight_dev.gold.dim_usuario

In [0]:
spark.sql('SELECT * FROM dataslight_dev.bronze.dados_database').display()
spark.sql('SELECT * FROM dataslight_dev.silver.tb_curso').display()
spark.sql('SELECT * FROM dataslight_dev.gold.dim_curso').display()

In [0]:
generate_database_data(type_of_data='delete row')

In [0]:
spark.sql('SELECT * FROM dataslight_dev.bronze.dados_database').display()
spark.sql('SELECT * FROM dataslight_dev.silver.tb_curso').display()
spark.sql('SELECT * FROM dataslight_dev.gold.dim_curso').display()

In [0]:
%sql SELECT * from dataslight_dev.gold.fato_venda

In [0]:
%sql
SELECT
    COUNT(*) AS quantdade_cursos_vendidos,
    SUM(f.valor_pago) - SUM(f.valor_desconto) AS valor_faturamento_bruto
FROM dataslight_dev.gold.fato_venda as f
JOIN dataslight_dev.gold.dim_codigos AS cd ON f.sk_codigos = cd.sk_codigos
JOIN dataslight_dev.gold.dim_curso AS cs ON f.sk_curso = cs.sk_curso 
JOIN dataslight_dev.gold.dim_tempo AS t ON f.sk_tempo = t.sk_tempo
JOIN dataslight_dev.gold.dim_usuario AS u ON f.sk_usuario = u.sk_usuario
WHERE t.ano = 2024
  AND t.mes = 'junho'
  AND cd.local_acesso = 'Mobile'
  AND cd.percentual_disconto = '5%'

In [0]:
%sql
SELECT
  SUM(qtde_acesso) AS quantidade_acessos,
  SUM(qtde_usuario) AS quantidade_usuario,
  SUM(qtde_vendidos) AS quantidade_vendas,
  CAST((SUM(qtde_vendidos)/SUM(qtde_acesso))*100 AS INT) AS percentual_vendas
FROM dataslight_dev.gold.obt_acesso
WHERE local_acesso = 'Computador'
  AND mes = 'junho'
  AND ano = 2024
  AND fl_dia_semana is false